In [1]:
print(1)

1


In [2]:
import mygene
import pandas as pd
import anndata as ad
import datasets
from datasets import load_dataset

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import ensembl_rest
import json
import os
from collections import Counter

In [4]:
from pathlib import Path
from collections import OrderedDict
from pathlib import Path
from typing import Iterable, Optional

def _read_table(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        gz = path.with_suffix(path.suffix + ".gz")
        if gz.exists():
            import gzip
            with gzip.open(gz, "rt") as fh:
                df = pd.read_csv(fh, **kwargs)
        else:
            raise FileNotFoundError(path)
    else:
        df = pd.read_csv(path, **kwargs)
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.lower()
    )
    return df

In [5]:
#!/usr/bin/env python
import time
import requests
from requests.exceptions import HTTPError
from typing import List, Dict, Any
from tqdm import tqdm


class EnsemblArchiveClient:
    """Simple client for Ensembl archive ID lookups with rate limiting"""
    
    def __init__(self, server: str = 'https://rest.ensembl.org', 
                       reqs_per_sec: int = 10,
                       n_retries: int = 10,
                       sleep: int = 10):
        """
        Initialize the client
        
        Args:
            server: Ensembl REST API server URL
            reqs_per_sec: Maximum requests per second (default 10)
        """
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0
        self.n_retries = n_retries
        self.sleep = sleep
    
    def _rate_limit(self):
        """Apply rate limiting to avoid overwhelming the server"""
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
    
    def get_archive_ids(self, ensembl_ids: List[str]) -> List[Dict[str, Any]]:
        """
        Look up archive information for Ensembl IDs
        
        Args:
            ensembl_ids: List of Ensembl gene IDs (e.g., ['ENSG00000251678', ...])
        
        Returns:
            List of dictionaries with archive information for each ID
        """
        self._rate_limit()
        
        url = f"{self.server}/archive/id"
        headers = {
            "Content-Type": "application/json",
            "Accept": "application/json"
        }
        payload = {"id": ensembl_ids}
        for i in range(self.n_retries):
            try:
                response = requests.post(url, headers=headers, json=payload)
                response.raise_for_status()
                self.req_count += 1
                return response.json()

            except HTTPError as e:
                print(f"Error fetching archive IDs: {e}")
                
                code = e.response.status_code
                if code == 429:
                    retry_after = int(response.headers.get('Retry-After', self.sleep))
                else:
                    retry_after = self.sleep
                print(f"Retrying {i} out of {self.n_retries}")
                time.sleep(retry_after)
        return [] 
                                
    
    def get_archive_ids_batch(self, ensembl_ids: List[str], batch_size: int = 1000, verbose: bool = True) -> List[Dict[str, Any]]:
        """
        Look up archive information in batches (useful for large lists)
        
        Args:
            ensembl_ids: List of Ensembl gene IDs
            batch_size: Number of IDs to query per request (default 1000)
            verbose: Show progress bar with tqdm (default True)
        
        Returns:
            Combined list of dictionaries with archive information
        """
        all_results = []
        
        # Create batches
        batches = [ensembl_ids[i:i + batch_size] for i in range(0, len(ensembl_ids), batch_size)]
        
        # Use tqdm if verbose, otherwise use plain iteration
        iterator = tqdm(batches, desc="Processing batches", unit="batch") if verbose else batches
        
        for batch in iterator:
            results = self.get_archive_ids(batch)
            all_results.extend(results)
        
        return all_results

In [6]:
def summarize_results(results, 
                      ens_ids=None, 
                      verbose=True):
    """
    Summarize Ensembl archive API results by creating a mapping dictionary
    and counting current, deprecated, and replaced gene IDs.
    
    Args:
        results: List of dictionaries from Ensembl archive API results
        verbose: If True, print warnings and summary statistics (default True)
        
    Returns:
        tuple: (mapping, cnt_current, cnt_deprecated, cnt_replaced, repl)
            - mapping: Dictionary mapping old IDs to current/replacement IDs
            - cnt_current: Count of current (non-deprecated) IDs
            - cnt_deprecated: Count of deprecated IDs with no replacement
            - cnt_replaced: Count of deprecated IDs with replacement(s)
            - repl: List of lists containing multiple replacements for genes
    """
    if not results:
        if verbose:
            print("Warning: Empty results list provided")
        return {}, 0, 0, 0, []
    
    mapping = {}
    cnt_current = 0
    cnt_deprecated = 0
    cnt_replaced_single = 0
    repl = []
    
    for item in results:
        if item['is_current'] == '1':
            if len(item['possible_replacement']) != 0:
                if verbose:
                    print(f"Warning, there are replacements {item['possible_replacement']} for existing ENS_ID {item['id']}")
            else:
                mapping[item['id']] = item['id']
                cnt_current += 1
        else:
            if len(item['possible_replacement']) == 0:
                cnt_deprecated += 1
            else:
                if len(item['possible_replacement']) > 1:
                    if verbose:
                        print(f"Warning, the gene {item['id']} has more than one replacements: {item['possible_replacement']}")
                    repl.append([r['stable_id'] for r in item['possible_replacement']])
                else:
                    mapping[item['id']] = item['possible_replacement'][0]['stable_id']
                    cnt_replaced_single += 1
    
    cnt_total = cnt_current + cnt_deprecated + cnt_replaced_single + len(repl)

    assert cnt_total == len(results)

    from collections import Counter
    duplicates = {k: v for k, v in Counter(mapping.values()).items() if v > 1}
        
    
    if verbose:
        print(f"\nSummary:")
        if not ens_ids is None:
            print(f"  Total results: {cnt_total} out of {len(ens_ids)}")
        else:
            print(f"  Total results: {cnt_total}")
        print(f"  Current IDs: {cnt_current}")
        print(f"  Deprecated (no replacement): {cnt_deprecated}")
        print(f"  Single replacements: {cnt_replaced_single}")
        print(f"  Multiple replacements: {len(repl)}")
        print(f"  Duplicated target IDs: {len(duplicates)}")
        print(f"  Mapping size: {len(mapping)}")
    
    return mapping, repl

In [7]:
def save_results(results, path):
    directory = os.path.dirname(path)
    os.makedirs(directory, exist_ok=True)
    with open(path, "w") as f:
        json.dump(results, f)

In [8]:
client = EnsemblArchiveClient()

# L1000

In [9]:
adata_l1000 = ad.read_h5ad('./lincs_data/processed/l1000_level3_deg_ready_landmark.h5ad')

In [10]:
file_path = './ens_ids/l1000_landmark.json'
if not os.path.isfile(file_path):
    results_l1000_landmark = client.get_archive_ids_batch(list(adata_l1000.var.index))
    save_results(results_l1000_landmark, file_path)
else:
    with open(file_path, "r") as f:
        results_l1000_landmark = json.load(f)

In [11]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_ROOT = (PROJECT_ROOT / "lincs_data").resolve()

PATHS = OrderedDict({
    "level3_gctx": DATA_ROOT / "GSE92742_Broad_LINCS_Level3_INF_mlr12k_n1319138x12328.gctx",
    "level3_gctx_gz": DATA_ROOT / "GSE92742_Broad_LINCS_Level3_INF_mlr12k_n1319138x12328.gctx.gz",
    "instinfo": DATA_ROOT / "GSE92742_Broad_LINCS_inst_info.txt",
    "cellinfo": DATA_ROOT / "GSE92742_Broad_LINCS_cell_info.txt",
    "pert_info": DATA_ROOT / "GSE92742_Broad_LINCS_pert_info.txt",
    "geneinfo_level3": DATA_ROOT / "GSE92742_Broad_LINCS_gene_info.txt",
    "geneinfo_beta": DATA_ROOT / "geneinfo_beta.txt",
    "cellinfo_beta": DATA_ROOT / "cellinfo_beta.txt",
    "compoundinfo_mw": DATA_ROOT / "compoundinfo_beta_with_MW.tsv",
})

l1000_all = _read_table(PATHS["geneinfo_beta"], sep="	") if PATHS["geneinfo_beta"].exists() else None

In [12]:
file_path = './ens_ids/l1000_all.json'
if not os.path.isfile(file_path):
    results_l1000_all = client.get_archive_ids_batch(list(l1000_all['ensembl_id'].values))
    save_results(results_l1000_all, file_path)
else:
    with open(file_path, "r") as f:
        results_l1000_all = json.load(f)

In [13]:
mapping_l1000_landmark, _ = summarize_results(results_l1000_landmark, list(adata_l1000.var.index))


Summary:
  Total results: 978 out of 978
  Current IDs: 978
  Deprecated (no replacement): 0
  Single replacements: 0
  Multiple replacements: 0
  Duplicated target IDs: 0
  Mapping size: 978


In [14]:
mapping_l1000_all, _ = summarize_results(results_l1000_all, list(l1000_all['ensembl_id'].values))

Warning, the gene ENSG00000203812 has more than one replacements: [{'stable_id': 'ENSG00000288859', 'score': 0.958449}, {'stable_id': 'ENSG00000288825', 'score': 0.959544}]

Summary:
  Total results: 12277 out of 12328
  Current IDs: 12261
  Deprecated (no replacement): 10
  Single replacements: 5
  Multiple replacements: 1
  Duplicated target IDs: 0
  Mapping size: 12265


# Sci-plex

In [15]:
adata_sciplex = ad.read_h5ad('./op3_v2/data/sciplex/pseudobulk/full/srivatsan20_sciplex3.h5ad')

In [16]:
file_path = './ens_ids/sciplex.json'
if not os.path.isfile(file_path):
    results_sciplex = client.get_archive_ids_batch(list(adata_sciplex.var.index))
    save_results(results_sciplex, file_path)
else:
    with open(file_path, "r") as f:
        results_sciplex = json.load(f)

In [17]:
mapping_sciplex, _ = summarize_results(results_sciplex, list(adata_sciplex.var.index))

Warning, the gene ENSG00000243135 has more than one replacements: [{'stable_id': 'ENSG00000288702', 'score': 0.986504}, {'stable_id': 'ENSG00000288705', 'score': 0.986027}]
Warning, the gene ENSG00000272196 has more than one replacements: [{'stable_id': 'ENSG00000288825', 'score': 0.971741}, {'score': 0.973404, 'stable_id': 'ENSG00000288859'}]
Warning, the gene ENSG00000244693 has more than one replacements: [{'stable_id': 'ENSG00000288784', 'score': 0.994811}, {'stable_id': 'ENSG00000289604', 'score': 0.994837}]
Warning, the gene ENSG00000225932 has more than one replacements: [{'score': 0.994837, 'stable_id': 'ENSG00000288784'}, {'score': 0.994811, 'stable_id': 'ENSG00000289604'}]
Warning, the gene ENSG00000277203 has more than one replacements: [{'stable_id': 'ENSG00000288722', 'score': 0.998249}, {'score': 0.998246, 'stable_id': 'ENSG00000288709'}]
Warning, the gene ENSG00000256374 has more than one replacements: [{'stable_id': 'ENSG00000288867', 'score': 0.788845}, {'stable_id': '

# Tahoe

In [18]:
adata_tahoe_lamin = ad.read_h5ad('./op3_v2/data/tahoe/pseudobulk/full/tahoe.h5ad')

In [19]:
adata_tahoe_lamin.var.reset_index()['ensembl_id'][~adata_tahoe_lamin.var.reset_index()['ensembl_id'].str.startswith('ENS')]

49              Y_RNA-61
142            Y_RNA-298
164            Y_RNA-673
248            Y_RNA-358
349            Y_RNA-565
              ...       
62604    Metazoa_SRP-162
62638          SNORA62-4
62656          HERC2P9-1
62672          Y_RNA-201
62676              7SK-6
Name: ensembl_id, Length: 1455, dtype: object

In [20]:
file_path = './ens_ids/tahoe.json'
if not os.path.isfile(file_path):
    results_tahoe_lamin = client.get_archive_ids_batch(list(adata_tahoe_lamin.var.index))
    save_results(results_tahoe_lamin, file_path)
else:
    with open(file_path, "r") as f:
        results_tahoe_lamin = json.load(f)

In [21]:
gene_metadata = load_dataset("vevotx/Tahoe-100M", name="gene_metadata", split="train")
gene_vocab = {'gene_symbol': [entry["gene_symbol"] for entry in gene_metadata], 
             'ensembl_id': [entry["ensembl_id"] for entry in gene_metadata], }
df_hf = pd.DataFrame(gene_vocab)
df_hf[~df_hf['ensembl_id'].str.startswith('ENS')]

,gene_symbol,ensembl_id


In [22]:
df_hf[df_hf['gene_symbol'] == 'Y_RNA-1']

,gene_symbol,ensembl_id
18019,Y_RNA-1,ENSG00000199200


In [23]:
file_path = './ens_ids/tahoe_hf.json'
if not os.path.isfile(file_path):
    results_tahoe_hf = client.get_archive_ids_batch(list(df_hf['ensembl_id'].values))
    save_results(results_tahoe_hf, file_path)
else:
    with open(file_path, "r") as f:
        results_tahoe_hf = json.load(f)

In [24]:
mapping_tahoe_lamin, _ = summarize_results(results_tahoe_lamin, list(adata_tahoe_lamin.var.index))


Summary:
  Total results: 61241 out of 62710
  Current IDs: 59942
  Deprecated (no replacement): 222
  Single replacements: 1077
  Multiple replacements: 0
  Duplicated target IDs: 833
  Mapping size: 61019


In [25]:
mapping_tahoe_hf, _ = summarize_results(results_tahoe_hf, list(df_hf['ensembl_id'].values))


Summary:
  Total results: 62705 out of 62710
  Current IDs: 61455
  Deprecated (no replacement): 165
  Single replacements: 1085
  Multiple replacements: 0
  Duplicated target IDs: 859
  Mapping size: 62540


# Compare LaminDB and HuggingFace gene annotation for Tahoe

In [26]:
def compare_ensembl_ids(df1, df2, 
                       df1_name_col='gene_symbol', 
                       df1_id_col='ensembl_id',
                       df2_name_col='gene_name',
                       df2_id_col='ensembl_gene_id',
                       col_names=['gene_symbol', 'df1_ensembl_id', 'df2_ensembl_id'],
                       verbose=True):
    """
    Compare Ensembl IDs between two dataframes and return mismatches as a DataFrame.
    
    Args:
        df1: First DataFrame with gene symbols and Ensembl IDs
        df2: Second DataFrame with gene names and Ensembl IDs
        df1_name_col: Column name in df1 containing gene symbols/names (default: 'gene_symbol')
        df1_id_col: Column name in df1 containing Ensembl IDs (default: 'ensembl_id')
        df2_name_col: Column name in df2 containing gene names. 
                      If None, uses df2.index (default: None)
        df2_id_col: Column name in df2 containing Ensembl IDs (default: 'ensembl_gene_id')
        verbose: If True, print summary statistics
    
    Returns:
        pd.DataFrame with columns: [df1_name_col, df1_id_col, df2_id_col]
        containing only rows where Ensembl IDs differ
    """
    # Create dictionaries from both dataframes
    names = df1[df1_name_col].values
    ens_ids = df1[df1_id_col].values
    d = {}
    for i in range(len(names)):
        d[names[i]] = ens_ids[i]
    
    if df2_name_col is None:
        # Use index as gene names
        names_df2 = df2.index.values
    else:
        names_df2 = df2[df2_name_col].values
    ens_ids_df2 = df2[df2_id_col].values
    d_df2 = {}
    for i in range(len(names_df2)):
        d_df2[names_df2[i]] = ens_ids_df2[i]
    
    # Find mismatches
    mismatches = []
    for key in d.keys():
        if key in d_df2 and d[key] != d_df2[key]:
            mismatches.append({
                col_names[0]: key,
                col_names[1]: d[key],
                col_names[2]: d_df2[key]
            })
    
    result_df = pd.DataFrame(mismatches)
    
    if verbose:
        print(f"Total genes compared: {len(d)}")
        print(f"Genes found in both datasets: {len(set(d.keys()) & set(d_df2.keys()))}")
        print(f"Mismatches in ENS_ids found: {len(result_df)}")
    
    return result_df

In [27]:
df_lamin = pd.read_parquet('./data/tahoe/raw/var.parquet')

In [28]:
compare_ensembl_ids(df_hf, df_lamin.reset_index(), col_names=['gene_symbol', 'ens_id_hf', 'ens_id_lamin'])

Total genes compared: 62710
Genes found in both datasets: 62710
Mismatches in ENS_ids found: 4036


,gene_symbol,ens_id_hf,ens_id_lamin
0,PRSS22,ENSG00000005001,ENSG00000282937
1,PRKAR2B,ENSG00000005249,ENSG00000284096
2,IBTK,ENSG00000005700,ENSG00000283068
3,YBX2,ENSG00000006047,ENSG00000288504
4,KRT33A,ENSG00000006059,ENSG00000261986
...,...,...,...
4031,SMG1P7-1,ENSG00000291263,SMG1P7-1
4032,SMG1P5-1,ENSG00000291266,SMG1P5-1
4033,ANKRD20A11P-1,ENSG00000291280,ANKRD20A11P-1
4034,COL6A4P1-1,ENSG00000291281,COL6A4P1-1


# Analysis of the results

In [29]:
def overlap_multi_replacements_against_mappings(results_by_dataset, mapping_by_dataset=None, verbose=True):
    """
    For each dataset, take genes that have MULTIPLE possible replacements and
    check how many of those replacement IDs appear in the mapped IDs of other datasets
    (using the newest/current Ensembl IDs from their mappings).

    Args:
        results_by_dataset: dict name -> results list (from Ensembl API)
        mapping_by_dataset: optional dict name -> mapping dict (old_id -> current/replacement id).
            If not provided, mapped IDs are derived directly from results:
            current IDs are kept; deprecated IDs contribute all possible_replacement IDs.
        verbose: if True, print a brief summary

    Returns:
        dict with keys:
            - multi_info: per dataset info with multi_repl_ids and gene->repl list
            - mapping_ids: per dataset set of mapped IDs (values of mapping)
            - overlaps: mapping of (src_with_multi, tgt_mapping) -> {
                  'count': int,
                  'ids': set of overlapping replacement IDs,
                  'genes': dict gene_id -> list of overlapping replacement IDs
              }
    """
    # collect multi-replacement info per dataset
    multi_info = {}
    for name, results in results_by_dataset.items():
        gene_to_reps = {}
        multi_repl_ids = set()
        for item in results:
            reps = item.get('possible_replacement') or []
            if len(reps) > 1:
                rep_ids = [r['stable_id'] for r in reps]
                gene_to_reps[item['id']] = rep_ids
                multi_repl_ids.update(rep_ids)
        multi_info[name] = {
            'genes': gene_to_reps,            # gene_id -> list of replacement IDs
            'multi_repl_ids': multi_repl_ids, # all replacement IDs for multi-replaced genes
        }

    # collect mapped (current) IDs per dataset
    if mapping_by_dataset is not None:
        mapping_ids = {name: set(mapping.values()) for name, mapping in mapping_by_dataset.items()}
    else:
        mapping_ids = {}
        for name, results in results_by_dataset.items():
            ids = set()
            for item in results:
                if item.get('is_current') == '1':
                    ids.add(item['id'])
                else:
                    reps = item.get('possible_replacement') or []
                    ids.update(r['stable_id'] for r in reps)
            mapping_ids[name] = ids

    overlaps = {}
    src_names = list(results_by_dataset.keys())
    tgt_names = list(mapping_ids.keys())

    for src in src_names:
        for tgt in tgt_names:
            if src.split('_')[0] == tgt.split('_')[0]:
                continue  # skip self
            overlap_ids = multi_info[src]['multi_repl_ids'].intersection(mapping_ids[tgt])
            if overlap_ids:
                # find which genes contributed overlapping replacement IDs
                gene_hits = {}
                for gene_id, rep_ids in multi_info[src]['genes'].items():
                    hits = [rid for rid in rep_ids if rid in overlap_ids]
                    if hits:
                        gene_hits[gene_id] = hits
            else:
                gene_hits = {}

            # compute overlap tuples mirroring the collect_overlaps helper (if available)
            try:
                # Pass as list of (gene_id, rep_ids) tuples for better gene tracking
                repl_list_with_genes = [(gene_id, rep_ids) for gene_id, rep_ids in multi_info[src]['genes'].items()]
                overlap_tuples_set, overlap_to_genes = collect_overlaps(
                    repl_list_with_genes, 
                    mapping_ids[tgt], 
                    gene_to_reps=None,  # Not needed when using tuple format
                    verbose=False
                )
            except NameError:
                # fallback inline to avoid dependency on definition order
                overlap_tuples_set = set()
                overlap_to_genes = {}
                target_set = set(mapping_ids[tgt])
                for gene_id, rep_ids in multi_info[src]['genes'].items():
                    overlap = target_set.intersection(set(rep_ids))
                    if overlap:
                        overlap_tuple = tuple(sorted(overlap))
                        overlap_tuples_set.add(overlap_tuple)
                        if overlap_tuple not in overlap_to_genes:
                            overlap_to_genes[overlap_tuple] = []
                        overlap_to_genes[overlap_tuple].append(gene_id)

            overlaps[(src, tgt)] = {
                'count': len(overlap_ids),
                'ids': overlap_ids,
                'genes': gene_hits,
                'overlap_tuples': overlap_tuples_set,
                'overlap_to_genes': overlap_to_genes,  # maps overlap_tuple -> list of gene_ids
            }

    if verbose:
        print("\nOverlap of multi-replacement IDs vs mapped IDs in other datasets")
        for (src, tgt), info in overlaps.items():
            if (info['count'] > 0) or len(info['genes']) > 0 or len(info['overlap_tuples']) > 0:
                print(f"\n> {src} (multi) vs {tgt} (mapped):")
                print(f"  - {info['count']} overlapping replacement IDs")
                print(f"  - {len(info['genes'])} genes with overlapping replacements")
                print(f"  - {len(info['overlap_tuples'])} unique overlap tuples")
                if info['overlap_to_genes']:
                    print(f"  - Associated genes and overlap tuples:")
                    for overlap_tuple, gene_list in info['overlap_to_genes'].items():
                        print(f"    {gene_list}: {overlap_tuple}")
            else:
                print(f"\n> {src} (multi) vs {tgt} (mapped) has no overlappings")

    return {
        'multi_info': multi_info,
        'mapping_ids': mapping_ids,
        'overlaps': overlaps,
    }


In [30]:
def collect_overlaps(repl_list, target_ids, gene_to_reps=None, verbose=True):
    """
    Given a list of replacement-ID lists (repl_list) and a set/list of target IDs,
    return sorted tuples of overlaps, mirroring the previous pattern.
    
    Args:
        repl_list: List of lists of replacement IDs (or list of tuples of (gene_id, rep_ids))
        target_ids: Set or list of target IDs to check against
        gene_to_reps: Optional dict mapping gene_id -> list of replacement IDs.
            If provided, will track which genes are associated with each overlap tuple.
            If repl_list contains tuples of (gene_id, rep_ids), this can be None.
        verbose: If True, print overlaps and associated genes
    
    Returns:
        tuple: (overlap_set, overlap_to_genes_dict)
            - overlap_set: Set of sorted tuples of overlapping IDs
            - overlap_to_genes: Dict mapping overlap_tuple -> list of gene_ids
    """
    target_set = set(target_ids)
    overlappings = []
    overlap_to_genes = {}
    
    # Check if repl_list contains tuples (gene_id, rep_ids) or just rep_ids
    if repl_list and isinstance(repl_list[0], tuple) and len(repl_list[0]) == 2:
        # repl_list is [(gene_id, rep_ids), ...]
        for gene_id, rep_ids in repl_list:
            overlap = target_set.intersection(set(rep_ids))
            overlap_tuple = tuple(sorted(overlap))
            
            if overlap_tuple:
                overlappings.append(overlap_tuple)
                if overlap_tuple not in overlap_to_genes:
                    overlap_to_genes[overlap_tuple] = []
                overlap_to_genes[overlap_tuple].append(gene_id)
                
                if verbose:
                    print(f"Associated genes: {gene_id}, overlap: {overlap}")
                    
    else:
        # repl_list is [rep_ids, ...]
        for rep_ids in repl_list:
            overlap = target_set.intersection(set(rep_ids))
            overlap_tuple = tuple(sorted(overlap))
            
            if overlap_tuple:
                overlappings.append(overlap_tuple)
                
                # Track which genes are associated with this overlap
                if gene_to_reps is not None:
                    # Find genes that have this exact set of replacement IDs
                    for gene_id, gene_reps in gene_to_reps.items():
                        if set(gene_reps) == set(rep_ids):
                            if overlap_tuple not in overlap_to_genes:
                                overlap_to_genes[overlap_tuple] = []
                            overlap_to_genes[overlap_tuple].append(gene_id)
                
                if verbose:
                    print(f"Associated genes: {overlap_to_genes[overlap_tuple]}, overlap: {overlap}")
    
    return set(overlappings), overlap_to_genes

# Example usage (uncomment and adjust variables as needed):
# overlappings, gene_map = collect_overlaps(repl, res_ids_lamin, gene_to_reps=gene_to_reps_dict)



In [31]:
def map_multiple_replacements(results, strategy='all', verbose=True):
    """
    Create a mapping for genes with multiple possible replacements.
    
    Args:
        results: List of dictionaries from Ensembl archive API results
        strategy: How to handle multiple replacements:
            - 'all': Map to all possible replacements (returns list of IDs)
            - 'first': Map to first replacement only
            - 'highest_score': Map to replacement with highest score (if available)
        verbose: If True, print summary
    
    Returns:
        dict: Mapping from old_id -> replacement_id(s) based on strategy
    """
    mapping = {}
    multi_repl_genes = []
    
    for item in results:
        reps = item.get('possible_replacement') or []
        if len(reps) > 1:
            multi_repl_genes.append(item['id'])
            
            if strategy == 'all':
                mapping[item['id']] = [r['stable_id'] for r in reps]
            elif strategy == 'first':
                mapping[item['id']] = reps[0]['stable_id']
            elif strategy == 'highest_score':
                # Sort by score (descending) and take the first
                sorted_reps = sorted(reps, key=lambda x: x.get('score', 0), reverse=True)
                mapping[item['id']] = sorted_reps[0]['stable_id']
            else:
                raise ValueError(f"Unknown strategy: {strategy}")
    
    if verbose:
        print(f"Found {len(multi_repl_genes)} genes with multiple replacements")
        print(f"Strategy '{strategy}' applied")
    
    return mapping



In [32]:
results_per_dataset = {'sciplex': results_sciplex,
                       'tahoe_lamin': results_tahoe_lamin,
                       'tahoe_hf': results_tahoe_hf,
                       'l1000_landmark': results_l1000_landmark,
                       'l1000_all': results_l1000_all}

In [33]:
_ = overlap_multi_replacements_against_mappings(results_per_dataset)


Overlap of multi-replacement IDs vs mapped IDs in other datasets

> sciplex (multi) vs tahoe_lamin (mapped):
  - 23 overlapping replacement IDs
  - 20 genes with overlapping replacements
  - 11 unique overlap tuples
  - Associated genes and overlap tuples:
    ['ENSG00000243135', 'ENSG00000240224']: ('ENSG00000288702', 'ENSG00000288705')
    ['ENSG00000272196', 'ENSG00000203812']: ('ENSG00000288825', 'ENSG00000288859')
    ['ENSG00000244693', 'ENSG00000225932']: ('ENSG00000288784', 'ENSG00000289604')
    ['ENSG00000277203', 'ENSG00000274791']: ('ENSG00000288709', 'ENSG00000288722')
    ['ENSG00000256374', 'ENSG00000263464']: ('ENSG00000288867', 'ENSG00000289549')
    ['ENSG00000183791']: ('ENSG00000288607', 'ENSG00000288616', 'ENSG00000288631')
    ['ENSG00000283401', 'ENSG00000261277']: ('ENSG00000284892', 'ENSG00000285116')
    ['ENSG00000283137', 'ENSG00000258786']: ('ENSG00000284788', 'ENSG00000285405')
    ['ENSG00000283336', 'ENSG00000258652']: ('ENSG00000285135', 'ENSG000002854

In [34]:
def compare_mapped_genes_across_datasets(results_by_dataset, mapping_by_dataset=None, verbose=True):
    """
    Overlap and compare all current/mapped genes across multiple datasets.
    
    For each dataset, collects all current Ensembl IDs (either from mappings or derived from results),
    then computes pairwise overlaps, common genes, and unique genes.
    
    Args:
        results_by_dataset: dict name -> results list (from Ensembl API)
        mapping_by_dataset: optional dict name -> mapping dict (old_id -> current/replacement id).
            If not provided, mapped IDs are derived directly from results:
            current IDs are kept; deprecated IDs contribute all possible_replacement IDs.
        verbose: if True, print detailed comparison statistics
    
    Returns:
        dict with keys:
            - mapped_ids: dict dataset_name -> set of mapped Ensembl IDs
            - pairwise_overlap: dict (dataset_a, dataset_b) -> set of overlapping IDs
            - pairwise_overlap_count: dict (dataset_a, dataset_b) -> count of overlaps
            - common_to_all: set of IDs present in all datasets
            - unique_per_dataset: dict dataset_name -> set of IDs unique to that dataset
            - summary_stats: dict with various statistics
    """
    from itertools import combinations
    
    # Collect mapped IDs for each dataset
    mapped_ids = {}
    
    if mapping_by_dataset is not None:
        # Use provided mappings
        for name, mapping in mapping_by_dataset.items():
            mapped_ids[name] = set(mapping.values())
    else:
        # Derive mapped IDs from results
        for name, results in results_by_dataset.items():
            ids = set()
            for item in results:
                if item.get('is_current') == '1':
                    ids.add(item['id'])
                else:
                    reps = item.get('possible_replacement') or []
                    if len(reps) == 1:
                        ids.add(reps[0]['stable_id'])
                    elif len(reps) > 1:
                        # For multiple replacements, include all of them
                        ids.update(r['stable_id'] for r in reps)
            mapped_ids[name] = ids
    
    dataset_names = list(mapped_ids.keys())
    
    # Compute pairwise overlaps
    pairwise_overlap = {}
    pairwise_overlap_count = {}
    
    for a, b in combinations(dataset_names, 2):
        overlap = mapped_ids[a].intersection(mapped_ids[b])
        pairwise_overlap[(a, b)] = overlap
        pairwise_overlap_count[(a, b)] = len(overlap)
    
    # Find genes common to all datasets
    if len(dataset_names) > 1:
        common_to_all = set.intersection(*[mapped_ids[name] for name in dataset_names])
    else:
        common_to_all = set()
    
    # Find unique genes per dataset (present only in that dataset)
    unique_per_dataset = {}
    for name in dataset_names:
        other_datasets = [mapped_ids[other] for other in dataset_names if other != name]
        if other_datasets:
            union_of_others = set.union(*other_datasets) if other_datasets else set()
            unique_per_dataset[name] = mapped_ids[name] - union_of_others
        else:
            unique_per_dataset[name] = mapped_ids[name]
    
    
    
    summary_stats = {
        'num_datasets': len(dataset_names),
        'common_to_all_count': len(common_to_all),
        'unique_per_dataset_count': {name: len(unique_per_dataset[name]) for name in dataset_names},
        'dataset_sizes': {name: len(mapped_ids[name]) for name in dataset_names},
    }
    
    if verbose:
        print("=" * 60)
        print("Comparison of Mapped Genes Across Datasets")
        print("=" * 60)
        print(f"\nNumber of datasets: {len(dataset_names)}")
        print(f"Genes common to all datasets: {len(common_to_all)}")
        
        print(f"\nDataset sizes:")
        for name in dataset_names:
            print(f"  {name}: {len(mapped_ids[name])} mapped genes")
        
        print(f"\nUnique genes per dataset:")
        for name in dataset_names:
            print(f"  {name}: {len(unique_per_dataset[name])} unique genes")
        
        if len(dataset_names) > 1:
            print(f"\nPairwise overlaps:")
            for (a, b), count in sorted(pairwise_overlap_count.items()):
                overlap_pct_a = (count / len(mapped_ids[a])) * 100 if len(mapped_ids[a]) > 0 else 0
                overlap_pct_b = (count / len(mapped_ids[b])) * 100 if len(mapped_ids[b]) > 0 else 0
                print(f"  {a} ∩ {b}: {count} genes ({overlap_pct_a:.1f}% of {a}, {overlap_pct_b:.1f}% of {b})")
        
        if common_to_all:
            print(f"\nGenes common to all datasets: {len(common_to_all)}")
            if len(common_to_all) <= 20:
                print(f"  Sample: {sorted(list(common_to_all))[:10]}")
            else:
                print(f"  Sample (first 10): {sorted(list(common_to_all))[:10]}")
    
    return {
        'mapped_ids': mapped_ids,
        'pairwise_overlap': pairwise_overlap,
        'pairwise_overlap_count': pairwise_overlap_count,
        'common_to_all': common_to_all,
        'unique_per_dataset': unique_per_dataset,
        'summary_stats': summary_stats,
    }



In [35]:
output = compare_mapped_genes_across_datasets(results_per_dataset)

Comparison of Mapped Genes Across Datasets

Number of datasets: 5
Genes common to all datasets: 926

Dataset sizes:
  sciplex: 56823 mapped genes
  tahoe_lamin: 59993 mapped genes
  tahoe_hf: 61483 mapped genes
  l1000_landmark: 978 mapped genes
  l1000_all: 12267 mapped genes

Unique genes per dataset:
  sciplex: 4 unique genes
  tahoe_lamin: 2213 unique genes
  tahoe_hf: 15 unique genes
  l1000_landmark: 0 unique genes
  l1000_all: 0 unique genes

Pairwise overlaps:
  l1000_landmark ∩ l1000_all: 978 genes (100.0% of l1000_landmark, 8.0% of l1000_all)
  sciplex ∩ l1000_all: 12266 genes (21.6% of sciplex, 100.0% of l1000_all)
  sciplex ∩ l1000_landmark: 978 genes (1.7% of sciplex, 100.0% of l1000_landmark)
  sciplex ∩ tahoe_hf: 56819 genes (100.0% of sciplex, 92.4% of tahoe_hf)
  sciplex ∩ tahoe_lamin: 53132 genes (93.5% of sciplex, 88.6% of tahoe_lamin)
  tahoe_hf ∩ l1000_all: 12267 genes (20.0% of tahoe_hf, 100.0% of l1000_all)
  tahoe_hf ∩ l1000_landmark: 978 genes (1.6% of tahoe_hf

In [36]:
output['unique_per_dataset']

{'sciplex': {'ENSG00000233143',
  'ENSG00000289022',
  'ENSG00000289240',
  'ENSG00000289542'},
 'tahoe_lamin': {'ENSG00000282294',
  'ENSG00000237095',
  'ENSG00000280751',
  'ENSG00000282536',
  'ENSG00000285098',
  'ENSG00000282186',
  'ENSG00000274334',
  'ENSG00000236488',
  'ENSG00000280712',
  'ENSG00000281074',
  'ENSG00000283949',
  'ENSG00000282212',
  'ENSG00000281956',
  'ENSG00000282076',
  'ENSG00000247700',
  'ENSG00000282211',
  'ENSG00000285414',
  'ENSG00000281197',
  'ENSG00000285236',
  'ENSG00000273495',
  'ENSG00000235764',
  'ENSG00000288517',
  'ENSG00000275624',
  'ENSG00000285087',
  'ENSG00000280669',
  'ENSG00000281313',
  'ENSG00000281748',
  'ENSG00000288205',
  'ENSG00000288466',
  'ENSG00000281184',
  'ENSG00000282867',
  'ENSG00000262749',
  'ENSG00000284804',
  'ENSG00000275758',
  'ENSG00000282937',
  'ENSG00000276833',
  'ENSG00000284939',
  'ENSG00000236598',
  'ENSG00000284937',
  'ENSG00000285315',
  'ENSG00000275190',
  'ENSG00000274352',
  'ENSG

# Analysis of Duplicated Gene Mappings

This section contains functions to analyze cases where **multiple old gene IDs map to the same new gene ID** (duplicates).

In [37]:
mapping_per_dataset = {'sciplex': mapping_sciplex,
                       'tahoe_lamin': mapping_tahoe_lamin,
                       'tahoe_hf': mapping_tahoe_hf,
                       'l1000_landmark': mapping_l1000_landmark,
                       'l1000_all': mapping_l1000_all}

In [38]:
def analyze_duplicated_gene_mappings(mapping_by_dataset, verbose=True):
    """
    Comprehensive analysis of duplicated gene mappings combining functionality from both
    analyze_duplicated_gene_mappings and analyze_duplicate_intersection_detailed.
    
    Analyzes genes where multiple old IDs map to the same new ID (duplicates).
    Includes both high-level statistics and detailed intersection analysis.
    
    Args:
        mapping_by_dataset: dict dataset_name -> mapping dict (old_id -> new_id)
        verbose: if True, print detailed statistics
    
    Returns:
        dict with keys:
            - duplicates_info: per-dataset info about duplicate mappings (includes both analysis types)
            - overlap_duplicates: pairwise overlaps of duplicated target IDs
            - overlap_duplicates_count: counts of pairwise overlaps
            - common_duplicates: target IDs that are duplicated across all datasets
            - summary_stats: overall statistics
            - intersection_analysis: detailed intersection analysis per dataset
    """
    from itertools import combinations
    
    # Analyze duplicates for each dataset
    duplicates_info = {}
    intersection_analysis = {}
    
    for name, mapping in mapping_by_dataset.items():
        # Create DataFrame for analysis
        df = pd.DataFrame(list(mapping.items()), columns=["source_id", "target_id"])
        
        # Find duplicated target IDs (multiple sources -> same target)
        duplicated_targets = df[df['target_id'].duplicated(keep=False)]['target_id'].unique()
        
        # Create mapping-value pairs (for intersection analysis)
        mapping_set = set((key, mapping[key]) for key in mapping.keys())
        mapping_pairs = mapping_set  # Alias for compatibility
        
        # Create duplicate-target pairs (self-pairs for duplicated targets)
        duplicate_self_pairs = set((target, target) for target in duplicated_targets)
        
        # Find overlap between mapping pairs and duplicate self-pairs
        intersection = mapping_set.intersection(duplicate_self_pairs)
        overlap_count = len(intersection)
        
        # Get diff: duplicates not in original keys
        diff_duplicates = [item[1] for item in (duplicate_self_pairs - mapping_set)]
        
        # Get detailed info about which sources map to each duplicated target
        target_to_sources = {}
        for target in duplicated_targets:
            sources = df[df['target_id'] == target]['source_id'].tolist()
            target_to_sources[target] = sources
        
        # Get DataFrame of duplicated entries, sorted
        df_duplicates = df[df['target_id'].isin(duplicated_targets)].sort_values(['target_id', 'source_id'])
        
        # Store comprehensive duplicates info
        duplicates_info[name] = {
            'duplicated_targets': set(duplicated_targets),
            'num_duplicated_targets': len(duplicated_targets),
            'target_to_sources': target_to_sources,
            'mapping_pairs': mapping_pairs,
            'duplicate_self_pairs': duplicate_self_pairs,
            'overlap_count': overlap_count,
            'total_mappings': len(mapping),
            'df': df,
            'df_duplicates': df_duplicates
        }
        
        # Store detailed intersection analysis
        intersection_analysis[name] = {
            'mapping_set': mapping_set,
            'duplicate_self_pairs': duplicate_self_pairs,
            'intersection': intersection,
            'intersection_count': overlap_count,
            'duplicated_values': set(duplicated_targets),  # Same as duplicated_targets
            'diff_duplicates': diff_duplicates,
            'df_duplicates': df_duplicates,
            'df': df
        }
    
    # Compute pairwise overlaps of duplicated targets
    dataset_names = list(mapping_by_dataset.keys())
    overlap_duplicates = {}
    overlap_duplicates_count = {}
    
    for a, b in combinations(dataset_names, 2):
        overlap = duplicates_info[a]['duplicated_targets'].intersection(
            duplicates_info[b]['duplicated_targets']
        )
        overlap_duplicates[(a, b)] = overlap
        overlap_duplicates_count[(a, b)] = len(overlap)
    
    # Find duplicated targets common to all datasets
    if len(dataset_names) > 1:
        common_duplicates = set.intersection(*[
            duplicates_info[name]['duplicated_targets'] 
            for name in dataset_names
        ])
    else:
        common_duplicates = set()
    
    # Summary statistics
    summary_stats = {
        'num_datasets': len(dataset_names),
        'common_duplicates_count': len(common_duplicates),
        'duplicates_per_dataset': {
            name: info['num_duplicated_targets'] 
            for name, info in duplicates_info.items()
        },
        'total_mappings_per_dataset': {
            name: info['total_mappings']
            for name, info in duplicates_info.items()
        },
        'duplicate_percentage': {
            name: (info['num_duplicated_targets'] / info['total_mappings'] * 100) 
            if info['total_mappings'] > 0 else 0
            for name, info in duplicates_info.items()
        }
    }
    
    if verbose:
        print("=" * 70)
        print("Comprehensive Analysis of Duplicated Gene Mappings")
        print("=" * 70)
        print(f"\nNumber of datasets: {len(dataset_names)}")
        
        print(f"\nDuplicates per dataset (multiple old IDs → same new ID):")
        for name in dataset_names:
            info = duplicates_info[name]
            int_info = intersection_analysis[name]
            pct = summary_stats['duplicate_percentage'][name]
            print(f"  {name}:")
            print(f"    - {info['num_duplicated_targets']} duplicated targets ({pct:.2f}% of {info['total_mappings']} total mappings)")
            print(f"    - Intersection count (key,value) in mapping AND (value,value) in duplicates: {int_info['intersection_count']}")
            print(f"    - Duplicated values NOT in original keys: {len(int_info['diff_duplicates'])}")
            
            # Show examples
            if info['target_to_sources']:
                examples = list(info['target_to_sources'].items())[:3]
                for target, sources in examples:
                    print(f"      Example: {sources} → {target}")
            
            if len(int_info['diff_duplicates']) > 0 and len(int_info['diff_duplicates']) <= 5:
                print(f"      Examples of diff_duplicates: {int_info['diff_duplicates'][:5]}")
        
        if len(dataset_names) > 1:
            print(f"\nPairwise overlaps of duplicated targets:")
            for (a, b), count in sorted(overlap_duplicates_count.items()):
                
                if count >= 0:
                    pct_a = (count / duplicates_info[a]['num_duplicated_targets'] * 100) \
                        if duplicates_info[a]['num_duplicated_targets'] > 0 else 0
                    pct_b = (count / duplicates_info[b]['num_duplicated_targets'] * 100) \
                        if duplicates_info[b]['num_duplicated_targets'] > 0 else 0
                    print('\n')
                    print(f"  {a} ∩ {b}: {count} duplicated targets "
                          f"({pct_a:.1f}% of {a}, {pct_b:.1f}% of {b})")
                    
                    # Show examples of overlapping duplicates
                    if count > 0:
                        overlap = overlap_duplicates[(a, b)]
                        examples = list(overlap)[:3]
                        for target in examples:
                            sources_a = duplicates_info[a]['target_to_sources'][target]
                            sources_b = duplicates_info[b]['target_to_sources'][target]
                            print(f"    {target}: {a}={sources_a}, {b}={sources_b}")
        
        if common_duplicates:
            print(f"\nDuplicated targets common to all datasets: {len(common_duplicates)}")
            if len(common_duplicates) <= 10:
                print(f"  All: {sorted(list(common_duplicates))}")
            else:
                print(f"  Sample (first 10): {sorted(list(common_duplicates))[:10]}")
        else:
            print(f"\nNo duplicated targets common to all datasets")
        
        print("\n" + "=" * 70)
    
    return {
        'duplicates_info': duplicates_info,
        'overlap_duplicates': overlap_duplicates,
        'overlap_duplicates_count': overlap_duplicates_count,
        'common_duplicates': common_duplicates,
        'summary_stats': summary_stats,
        'intersection_analysis': intersection_analysis,
    }

In [39]:
duplicate_analysis = analyze_duplicated_gene_mappings(mapping_per_dataset, verbose=True)

Comprehensive Analysis of Duplicated Gene Mappings

Number of datasets: 5

Duplicates per dataset (multiple old IDs → same new ID):
  sciplex:
    - 604 duplicated targets (1.05% of 57502 total mappings)
    - Intersection count (key,value) in mapping AND (value,value) in duplicates: 592
    - Duplicated values NOT in original keys: 12
      Example: ['ENSG00000284657', 'ENSG00000284602'] → ENSG00000284657
      Example: ['ENSG00000259594', 'ENSG00000259343'] → ENSG00000259343
      Example: ['ENSG00000182912', 'ENSG00000235890'] → ENSG00000235890
  tahoe_lamin:
    - 833 duplicated targets (1.37% of 61019 total mappings)
    - Intersection count (key,value) in mapping AND (value,value) in duplicates: 830
    - Duplicated values NOT in original keys: 3
      Example: ['ENSG00000289084', 'ENSG00000233067', 'ENSG00000224204'] → ENSG00000233067
      Example: ['ENSG00000228933', 'ENSG00000242021'] → ENSG00000228933
      Example: ['ENSG00000147753', 'ENSG00000237563'] → ENSG00000147753
  

## Mapping and Aggregating Duplicated Genes in AnnData

When multiple old gene IDs map to the same new gene ID, you need to aggregate their counts.
This section provides functions to map gene IDs and aggregate counts for AnnData objects.

In [40]:
# Helper functions for aggregation
from tqdm import tqdm 
import numpy as np
from scipy import sparse

# OPTIMIZED VERSION - Replace the _aggregate_matrix function above with this:
# This version uses vectorized matrix multiplication and is orders of magnitude faster

def _aggregate_matrix(matrix, groups, new_id_to_index, var_index):
    """Aggregate matrix (X or layer) for duplicated genes using vectorized matrix multiplication.
    
    This is much faster than iterating through groups because it uses a single matrix multiplication
    instead of extracting columns one by one. Orders of magnitude faster for large matrices.
    """
    import numpy as np
    from scipy import sparse
    
    # Build aggregation matrix: maps old gene indices to new gene indices
    # Shape: (n_genes_old, n_genes_new)
    # Each row corresponds to an old gene, each column to a new gene
    # Value is 1 if old gene maps to new gene, 0 otherwise
    old_cols = []
    new_cols = []
    
    for new_id, group in groups:
        if pd.isna(new_id):
            continue
        new_col = new_id_to_index[new_id]
        for idx in group.index:
            old_cols.append(var_index.get_loc(idx))
            new_cols.append(new_col)
    
    # Create sparse aggregation matrix
    # Use matrix dtype for data to preserve precision
    data = np.ones(len(old_cols), dtype=matrix.dtype)
    n_genes_old = matrix.shape[1]
    n_genes_new = len(new_id_to_index)
    
    aggregation_matrix = sparse.csr_matrix(
        (data, (old_cols, new_cols)),
        shape=(n_genes_old, n_genes_new),
        dtype=matrix.dtype
    )
    
    # Matrix multiplication: matrix @ aggregation_matrix
    # This sums columns (old genes) that map to the same new gene in a single operation
    agg_matrix = matrix @ aggregation_matrix
    
    return agg_matrix

# To use: Replace _aggregate_matrix with _aggregate_matrix_fast in map_and_aggregate_duplicated_genes
# Or simply rename this function to _aggregate_matrix

In [43]:
def map_and_aggregate_duplicated_genes(adata, mapping, var_index_col=None,
                                                keep_unmapped=False, verbose=True):
    """Shorter version - same functionality, more concise code."""
    if var_index_col and var_index_col not in adata.var.columns:
        raise ValueError(f"Column '{var_index_col}' not found in adata.var")
    
    n_genes_init = adata.n_vars
    gene_ids = adata.var[var_index_col] if var_index_col else adata.var.index
    mapped_ids = np.array([mapping.get(gid, gid if keep_unmapped else None) for gid in gene_ids], dtype=object)
    
    if not keep_unmapped:
        mask = mapped_ids != None
        adata = adata[:, mask].copy()
        mapped_ids = mapped_ids[mask]
        gene_ids = gene_ids[mask]
    
    mapping_df = pd.DataFrame({'old_id': gene_ids.values, 'new_id': mapped_ids}, index=adata.var.index)
    groups = mapping_df.groupby('new_id')
    unique_new_ids = mapping_df['new_id'].dropna().unique()
    duplicated_targets = mapping_df[mapping_df['new_id'].duplicated(keep=False)]['new_id'].unique()
    
    if verbose:
        print(f"Total genes: {n_genes_init}, Mapped: {mapping_df['new_id'].notna().sum()}")
        print(f"The number of unique target IDs with existing duplicates: {len(duplicated_targets)}")
        if len(duplicated_targets) > 0:
            print(f"  Aggregating running...")
    
    new_id_to_index = {gid: idx for idx, gid in enumerate(unique_new_ids)}
    X_agg = _aggregate_matrix(adata.X, groups, new_id_to_index, adata.var.index)
    new_var = pd.DataFrame({
        col: [adata.var.loc[groups.get_group(new_id).index[0], col] for new_id in unique_new_ids]
        for col in adata.var.columns if col != var_index_col
    }, index=unique_new_ids)
    
    adata_mapped = ad.AnnData(X=X_agg, obs=adata.obs.copy(), var=new_var,
                               uns=getattr(adata, 'uns', {}).copy(),
                               obsm=getattr(adata, 'obsm', {}).copy(),
                               varm=getattr(adata, 'varm', {}).copy())
    
    if verbose:
        print(f"\nResult: {adata.n_vars} → {adata_mapped.n_vars} genes (reduced by {adata.n_vars - adata_mapped.n_vars})")
    return adata_mapped

In [44]:
adata_mapped = map_and_aggregate_duplicated_genes(
     adata_sciplex, 
     mapping_sciplex,
     keep_unmapped=False,        # Remove genes not in mapping
     verbose=True
 )

Total genes: 58302, Mapped: 57502
The number of unique target IDs with existing duplicates: 604
  Aggregating running...

Result: 57502 → 56799 genes (reduced by 703)
